<div align="center">

**Taller Pandas**

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/e/ed/Pandas_logo.svg/240px-Pandas_logo.svg.png" alt="Pandas" width="240"/>

</div>

# Taller práctico: Pandas con Refinitiv LSEG

## Objetivo

Conectarte a **Refinitiv LSEG** desde Python y realizar consultas de datos utilizando **pandas**. Los resultados de las APIs de Refinitiv se obtienen en formato que se puede convertir fácilmente a DataFrame para exploración y análisis.

**Requisitos previos:**
- Eikon 4.0.36+ o Refinitiv Workspace 1.8+ instalado y abierto en tu equipo, **o** credenciales de Refinitiv Data Platform (RDP).
- App Key de Refinitiv (para sesión Desktop) o usuario/contraseña para RDP.

En este taller practicarás:
- Conectar a Refinitiv LSEG (sesión Desktop o Platform)
- Obtener datos (precios históricos, fundamentales) y convertirlos a DataFrame
- Explorar DataFrames (`.head()`, `.info()`, `.shape`, columnas)
- Trabajar con Series, filtrar y seleccionar datos
- Índices de fechas y rangos

In [ ]:
import os

# Ruta a la carpeta Configuration del proyecto (donde está lseg-data.config.json).
# Usa ruta absoluta para que funcione sin importar desde dónde se ejecute el notebook.
_cwd = os.getcwd()
_config_dir = os.path.join(_cwd, "Configuration")
if os.path.isdir(_config_dir):
    os.environ["LD_LIB_CONFIG_PATH"] = os.path.abspath(_config_dir)
else:
    # Si Configuration no existe, usar la carpeta actual (p. ej. Notebooks, donde también puede estar el config)
    os.environ["LD_LIB_CONFIG_PATH"] = os.path.abspath(_cwd)

---

## Conexión a Refinitiv LSEG con archivo de configuración

La configuración sigue el **[Quick Start](https://github.com/LSEG-API-Samples/Example.DataLibrary.Python/tree/lseg-data-examples/Quick%20Start)** del repo [Example.DataLibrary.Python](https://github.com/LSEG-API-Samples/Example.DataLibrary.Python) (rama `lseg-data-examples`): credenciales en **`Configuration/lseg-data.config.json`**; no modificar el config interno del paquete.

**Pasos:**

1. **Refinitiv Workspace abierto** en tu equipo e inicia sesión.
2. Obtén tu **App Key** (Workspace → configuración / API Key o App Key). Al generarla, acepta los términos que muestra Refinitiv (es normal; no es un error).
3. **Edita el archivo de configuración** (no el código):
   - En esta misma carpeta (`Notebooks`) hay un archivo **`lseg-data.config.json`**.
   - O en la subcarpeta **`Configuration/`** (junto con un `README.md` que explica la estructura).
   - Abre `lseg-data.config.json` y reemplaza `"TU_APP_KEY"` por tu App Key real.
4. Ejecuta la celda siguiente: la librería leerá la config y abrirá sesión **Desktop** (usa la sesión ya abierta en Workspace; no hace falta usuario/contraseña en el archivo).

**Estructura del config (según [repo oficial LSEG](https://github.com/LSEG-API-Samples/Example.DataLibrary.Python/blob/lseg-data-examples/Configuration/lseg-data.config.json)):** Incluye `logs` y sesión Desktop como `desktop.workspace`.
```json
{
  "logs": { "level": "debug", "transports": { "console": { "enabled": false }, "file": { "enabled": false } } },
  "sessions": {
    "default": "desktop.workspace",
    "desktop": {
      "workspace": { "app-key": "TU_APP_KEY" }
    }
  }
}
```

**Importante:** No modifiques el archivo de configuración que viene dentro del paquete instalado (site-packages). Usa la copia en tu proyecto (`Notebooks/` o `Configuration/`).

**Uso conforme a los términos:** Uso individual (un usuario, una sesión). No compartas credenciales ni redistribuyas datos de Refinitiv de forma contraria al acuerdo de tu organización.

**Instalación:** `pip install lseg-data`

In [1]:
2+2

4

In [3]:
!pip install lseg-data

  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached pandas-2.3.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached scipy-1.17.0-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached tenacity-9.1.2-py3-none-any.whl.metadata (1.2 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached certifi-2026.1.4-py3-none-any.whl.metadata (2.5 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-win_amd64.whl.metadata (2.8 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
 

  You can safely remove it manually.

[notice] A new release of pip is available: 24.2 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
!pip install refinitiv

ERROR: Could not find a version that satisfies the requirement refinitiv (from versions: none)

[notice] A new release of pip is available: 24.2 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for refinitiv


In [2]:
import pandas as pd

In [5]:
import lseg

In [29]:
# Conexión a Refinitiv LSEG (Refinitiv Workspace)
# Usa ld.open_session(name='desktop.default', app_key=..., config_name=...) para evitar error de session_name

import os
import json
import pandas as pd

USE_REFINITIV = True  # Tienes acceso a Workspace: la conexión se activa por defecto

# Ruta al config (Notebooks/ o Configuration/) — ruta absoluta para que la librería lo encuentre
_cwd = os.getcwd()
_config_path = os.path.abspath(os.path.join(_cwd, "lseg-data.config.json"))
if not os.path.isfile(_config_path):
    _config_path = os.path.abspath(os.path.join(_cwd, "Configuration", "lseg-data.config.json"))

if USE_REFINITIV:
    try:
        with open(_config_path, "r", encoding="utf-8") as f:
            _config = json.load(f)
        _desktop = _config.get("sessions", {}).get("desktop", {})
        _app_key = (_desktop.get("workspace") or {}).get("app-key") or (_desktop.get("default") or {}).get("app-key")
        if not _app_key or _app_key == "TU_APP_KEY":
            raise ValueError("Edita lseg-data.config.json y pon tu App Key en 'desktop.workspace.app-key'.")

        import lseg.data as ld
        # Formato oficial LSEG: sesión Desktop = 'desktop.workspace' (ver repo Example.DataLibrary.Python)
        ld.open_session(name="desktop.workspace", app_key=_app_key, config_name=_config_path)
        print("Sesión Refinitiv (Workspace) abierta correctamente.")
    except Exception as e:
        print("No se pudo conectar a Refinitiv:", e)
        print("Comprueba: 1) Refinitiv Workspace abierto e iniciado sesión.")
        print("2) App Key correcta en Configuration/lseg-data.config.json (o Notebooks/lseg-data.config.json).")
        USE_REFINITIV = False
else:
    print("Sesión Refinitiv no activa (USE_REFINITIV=False). Usa URLs de respaldo en los ejercicios.")

Sesión Refinitiv (Workspace) abierta correctamente.


In [67]:
import os
os.environ["LD_LIB_CONFIG_PATH"] = os.path.abspath("Configuration")

In [68]:

ld.open_session()

<lseg.data.session.Definition object at 0x217f3196d20 {name='workspace'}>

In [69]:
# Extraer datos con LSEG — ejecuta primero la celda de conexión (no llames ld.open_session() aquí)
import lseg.data as ld
import pandas as pd

# Obtener precios de cierre (TR.PriceClose) para Apple en 2023
result = ld.get_data(
    universe=["AAPL.OQ"],
    fields=["TR.PriceClose"])

# Convertir a pandas DataFrame (si el resultado tiene .data.df, usarlo; si no, el resultado ya es el DataFrame)
df = result.data.df if hasattr(result, "data") and hasattr(result.data, "df") else result
if not isinstance(df, pd.DataFrame):
    df = pd.DataFrame(df) if df is not None else pd.DataFrame()

print("Dimensiones:", df.shape)
print("\nPrimeras filas:")
display(df.head(10))
print("\nÚltimas filas:")
display(df.tail(5))

Dimensiones: (1, 2)

Primeras filas:


,Instrument,Price Close
0,AAPL.OQ,270.01



Últimas filas:


,Instrument,Price Close
0,AAPL.OQ,270.01


In [70]:
from lseg.data.content import historical_pricing

In [71]:
response = historical_pricing.summaries.Definition(
 universe="EC.N",  # Ecopetrol ADR en NYSE
 interval=historical_pricing.Intervals.DAILY,
 count=90).get_data()

In [72]:
response

Response(is_success=True, raw=[[<Response [200 OK]>]], errors=[], closure=None, requests_count=1, _data_factory=<lseg.data.content._historical_data_provider.HistoricalDataFactoryMultiResponse object at 0x00000217F320F080>, _kwargs={'auto_retry': False, 'interval': <Intervals.DAILY: 'P1D'>, 'start': None, 'end': None, 'adjustments': None, 'sessions': None, 'count': 90, 'fields': [], 'extended_params': None, '__data_type__': <ContentType.HISTORICAL_PRICING_INTERDAY_SUMMARIES: 52>, '__content_type__': <ContentType.HISTORICAL_PRICING_INTERDAY_SUMMARIES: 52>, 'responses': [Response(is_success=True, raw=[<Response [200 OK]>], errors=[], closure=None, requests_count=1, _data_factory=<lseg.data.content._historical_response_factory.HistoricalResponseFactory object at 0x00000217F3C46C60>, _kwargs={'session': <lseg.data.session.Definition object at 0x217f3196d20 {name='workspace'}>, 'universe': 'EC.N', 'auto_retry': False, 'interval': <Intervals.DAILY: 'P1D'>, 'start': None, 'end': None, 'adjustm

In [73]:
df_refinitiv = response.data.df

In [74]:
df_refinitiv

EC.N,TRDPRC_1,HIGH_1,LOW_1,ACVOL_UNS,OPEN_PRC,BID,ASK,TRNOVR_UNS,VWAP,BLKCOUNT,BLKVOLUM,NUM_MOVES,TRD_STATUS,SALTIM,VWAP_VOL
Date,,,,,,,,,,,,,,,
2025-09-24,9.63,9.635,9.41,715583,9.41,9.62,9.63,6840687,9.5596,3,133414,1665,1,72600,715583
2025-09-25,9.53,9.67,9.52,674902,9.65,9.52,9.53,6476148,9.5957,3,139291,1592,1,72600,674902
2025-09-26,9.41,9.565,9.4,608552,9.51,9.41,9.42,5764357,9.4723,3,89837,1514,1,72600,608552
2025-09-29,9.18,9.29,9.155,490478,9.19,9.16,9.17,4507253,9.1895,2,147176,1212,1,72600,490478
2025-09-30,9.21,9.24,9.08,507635,9.12,9.21,9.22,4659099,9.178,1,59644,1359,1,72600,507635
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-27,13.08,13.26,12.75,1287823,12.75,13.08,13.09,16803800,13.0482,3,185361,3943,1,76200,1287823
2026-01-28,12.76,13.21,12.525,1616422,13.1,12.75,12.76,20578707,12.731,2,256589,4981,1,76200,1616422
2026-01-29,12.77,13.23,12.61,1484764,13.1,12.77,12.78,19042298,12.8251,4,159095,5289,1,76200,1484764


In [75]:
ld.close_session()

### Obtener datos de Refinitiv y convertirlos a pandas DataFrame

**Importante:** Ejecuta primero la **celda de conexión** anterior. Si en otra celda usas `ld.get_data()` o similar, **no vuelvas a llamar** `ld.open_session()`; la sesión ya está abierta.

Una vez conectado, puedes usar el **Content Layer** para precios históricos o datos fundamentales. El resultado tiene un atributo `.data.df` que es un **pandas DataFrame**.

**Ejemplo: precios históricos** (Historical Pricing Summaries). RIC de Ecopetrol en NYSE: `EC.N`; en Colombia: `ECOPETROL.CB`.

In [24]:
# Ejemplo: obtener precios históricos de Ecopetrol (EC.N = NYSE) y cargar en pandas
# Solo se ejecuta si USE_REFINITIV es True y la sesión está abierta

if USE_REFINITIV:
    try:
        from lseg.data.content import historical_pricing
        
        response = historical_pricing.summaries.Definition(
            universe="EC.N",  # Ecopetrol ADR en NYSE
            interval=historical_pricing.Intervals.DAILY,
            count=90
        ).get_data()
        
        df_refinitiv = response.data.df  # pandas DataFrame
        print("DataFrame desde Refinitiv (primeras filas):")
        display(df_refinitiv.head(10))
    except Exception as e:
        print("Error al obtener datos de Refinitiv:", e)
        df_refinitiv = None
else:
    df_refinitiv = None
    print("Salta esta celda o activa USE_REFINITIV para obtener datos desde LSEG.")

Salta esta celda o activa USE_REFINITIV para obtener datos desde LSEG.


## Ejercicio 1: Cargar y explorar una base de datos (Refinitiv o URL)

**Opción A (con Refinitiv):** Usa el DataFrame `df_refinitiv` obtenido en la celda anterior (precios históricos). Si no lo tienes, define `df = df_refinitiv.copy()` y trabaja con `df`.

**Opción B (sin Refinitiv):** Carga el listado de constituyentes del S&P 500 desde esta URL en un DataFrame `df`:
`https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv`

**Tareas (sobre el DataFrame `df` que elijas):**
1. Asegúrate de tener un DataFrame `df` (desde Refinitiv o desde la URL)
2. Muestra las dimensiones (`.shape`) y las primeras 10 filas (`.head(10)`)
3. Usa `.info()` para ver tipos de columnas
4. Lista los nombres de las columnas (`.columns`)

In [25]:
# Opción A: usar DataFrame de Refinitiv (si ya ejecutaste la celda de precios históricos)
# Opción B: cargar S&P 500 desde URL
df_refinitiv = globals().get("df_refinitiv", None)
if df_refinitiv is not None and not df_refinitiv.empty:
    df = df_refinitiv.copy()
    print("Usando DataFrame desde Refinitiv LSEG.")
else:
    url_sp500 = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv"
    df = pd.read_csv(url_sp500)
    print("Usando DataFrame desde URL (S&P 500).")

# Exploración con pandas
print("Dimensiones:", df.shape)
print("\nPrimeras 10 filas:")
display(df.head(10))
print("\nTipos de columnas (.info()):")
df.info()
print("\nNombres de columnas:", df.columns.tolist())


Usando DataFrame desde URL (S&P 500).
Dimensiones: (503, 8)

Primeras 10 filas:


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
5,ADBE,Adobe Inc.,Information Technology,Application Software,"San Jose, California",1997-05-05,796343,1982
6,AMD,Advanced Micro Devices,Information Technology,Semiconductors,"Santa Clara, California",2017-03-20,2488,1969
7,AES,AES Corporation,Utilities,Independent Power Producers & Energy Traders,"Arlington, Virginia",1998-10-02,874761,1981
8,AFL,Aflac,Financials,Life & Health Insurance,"Columbus, Georgia",1999-05-28,4977,1955
9,A,Agilent Technologies,Health Care,Life Sciences Tools & Services,"Santa Clara, California",2000-06-05,1090872,1999



Tipos de columnas (.info()):
<class 'pandas.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   Symbol                 503 non-null    str  
 1   Security               503 non-null    str  
 2   GICS Sector            503 non-null    str  
 3   GICS Sub-Industry      503 non-null    str  
 4   Headquarters Location  503 non-null    str  
 5   Date added             503 non-null    str  
 6   CIK                    503 non-null    int64
 7   Founded                503 non-null    str  
dtypes: int64(1), str(7)
memory usage: 31.6 KB

Nombres de columnas: ['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'Headquarters Location', 'Date added', 'CIK', 'Founded']


## Ejercicio 2: Series y columnas

Usando el DataFrame `df` del Ejercicio 1:

**Si cargaste S&P 500 (constituyentes):**
1. Extrae la columna de sectores (p. ej. `GICS Sector`) como una Series
2. Usa `.value_counts()` para contar cuántas empresas hay por sector
3. Muestra los 5 sectores con más empresas
4. Obtén la columna de símbolos (tickers) y muestra los primeros 15 valores

**Si cargaste precios desde Refinitiv:** extrae la columna de cierre (p. ej. `Close` o `CLOSE`) como Series y calcula `.min()`, `.max()` y `.mean()`.

In [ ]:
# Según el origen del DataFrame (Refinitiv precios vs S&P 500), las columnas cambian
if "GICS Sector" in df.columns:
    serie_sector = df["GICS Sector"]
    print("Conteo por sector (.value_counts()):")
    print(serie_sector.value_counts().head(5))
    print("\nPrimeros 15 símbolos:")
    print(df["Symbol"].head(15))
else:
    # DataFrame de precios Refinitiv: buscar columna de cierre (puede ser Close, CLOSE, etc.)
    col_close = [c for c in df.columns if "close" in c.lower() or "CLOSE" in c]
    if col_close:
        serie = df[col_close[0]]
        print("Columna de cierre como Series - min, max, mean:")
        print(serie.min(), serie.max(), serie.mean())
    else:
        print("Columnas disponibles:", df.columns.tolist())


## Ejercicio 3: Filtrar datos

Sobre el mismo DataFrame `df`:

**Si es S&P 500 (constituyentes):**
1. Filtra solo las filas cuyo sector sea "Information Technology"
2. ¿Cuántas empresas de tecnología hay?
3. Filtra el sector "Financials" y muestra solo columnas Símbolo y Nombre (Symbol, Security)
4. Crea un sub-DataFrame con las primeras 20 filas y 3 columnas (Symbol, Security, GICS Sector)

**Si es precios Refinitiv:** filtra por rango de fechas con `.loc[]` (si el índice es fecha) o por una columna de fecha, y muestra las últimas 10 filas.

In [ ]:
# Filtrar según tipo de DataFrame
if "GICS Sector" in df.columns:
    tech = df[df["GICS Sector"] == "Information Technology"]
    print("Empresas de Information Technology:", len(tech))
    financials = df[df["GICS Sector"] == "Financials"][["Symbol", "Security"]]
    display(financials.head())
    sub = df.head(20)[["Symbol", "Security", "GICS Sector"]]
    display(sub)
else:
    # Precios Refinitiv: últimas 10 filas (o filtrar por fecha si hay columna Date)
    display(df.tail(10))


## Ejercicio 4: Serie temporal (índice de fechas)

**Opción A (Refinitiv):** Si ya tienes `df_refinitiv` de precios históricos, usa la columna de fecha como índice (`set_index`) y filtra con `.loc['2024-01-01':]`.

**Opción B (URL):** Carga el PIB trimestral de EE.UU. desde FRED:
`https://fred.stlouisfed.org/graph/fredgraph.csv?id=GDP`

**Tareas:**
1. Lee el CSV con `parse_dates` en la primera columna (o usa el DataFrame de Refinitiv)
2. Asigna nombres claros a las columnas (p. ej. `fecha`, `pib`) si usas FRED
3. Usa `.set_index()` para poner la fecha como índice
4. Muestra `.head(10)` y `.tail(5)`
5. Filtra desde 2020 con `.loc['2020-01-01':]`

In [ ]:
# Opción A: usar precios Refinitiv con índice de fecha
# Opción B: cargar PIB desde FRED
df_ref = globals().get("df_refinitiv", None)
if df_ref is not None and not df_ref.empty:
    # Buscar columna de fecha en el DataFrame de Refinitiv
    date_cols = [c for c in df_ref.columns if "date" in c.lower() or "Date" in c]
    if date_cols:
        df_ts = df_ref.copy()
        df_ts[date_cols[0]] = pd.to_datetime(df_ts[date_cols[0]])
        df_ts = df_ts.set_index(date_cols[0])
    else:
        df_ts = df_ref.copy()
else:
    url_gdp = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=GDP"
    df_ts = pd.read_csv(url_gdp, parse_dates=[0])
    df_ts.columns = ["fecha", "pib_millones_usd"]
    df_ts = df_ts.set_index("fecha")

print("Head(10):")
display(df_ts.head(10))
print("Tail(5):")
display(df_ts.tail(5))
print("\nDesde 2020:")
display(df_ts.loc["2020-01-01":].head())


---

### Cerrar sesión Refinitiv (opcional)

Al terminar el taller, si abriste sesión con Refinitiv, puedes cerrarla con:

In [ ]:
# Solo si usaste sesión Refinitiv
if USE_REFINITIV:
    try:
        import lseg.data as ld
        ld.close_session()
        print("Sesión Refinitiv cerrada.")
    except Exception:
        pass

## Ejercicio 5: Crear Series y DataFrame desde cero

**Tareas:**
1. Crea una Series con los precios de cierre de 5 días: `[100, 102, 98, 105, 103]` e índice `['Lun','Mar','Mié','Jue','Vie']`
2. Crea un DataFrame con dos columnas: `['Fecha', 'Monto']` y al menos 4 filas de datos (por ejemplo, gastos o ingresos)
3. Muestra `.shape`, `.columns` e `.info()` del DataFrame creado

In [ ]:
# Escribe tu código aquí



## Ejercicio 6: Resumen

Elige una de las bases que cargaste (S&P 500 o PIB) y:

**Tareas:**
1. Obtén la columna numérica como Series y calcula `.min()`, `.max()`, `.mean()` (si aplica)
2. Para el DataFrame de S&P 500: ¿cuántas columnas son de tipo objeto/texto y cuántas numéricas? (usa `.dtypes`)
3. Muestra las últimas 10 filas del DataFrame de PIB con `.tail(10)`

In [ ]:
# Escribe tu código aquí

